Overall Accuracy - ASDIV

Normal:
CoT - 93.66
Standard - 88.63
Complex CoT - 91.70

Hypothesis:
CoT - 91.22
Standard - 92.68%
Complex CoT - 86.82

In [1]:
import openai
import re
import time
import json

import numpy as np

from tqdm import tqdm
from pprint import pprint
from tenacity import retry, stop_after_attempt, wait_chain, wait_fixed

import os
from openai import AzureOpenAI

import math

import re
import math
import traceback
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor

In [2]:
endpoint = "https://pankajaiml.openai.azure.com/"
model_name = "gpt-4o"
deployment = "gpt-4o"
subscription_key = "REDACTED_AZURE_OPENAI_KEY"
api_version = "2024-12-01-preview"

client = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=endpoint,
    api_key=subscription_key,
)

# Retry logic
@retry(wait=wait_chain(*[wait_fixed(3) for _ in range(3)] +
                       [wait_fixed(5) for _ in range(2)] +
                       [wait_fixed(10)]))
def completion_with_backoff(messages):
    return client.chat.completions.create(
        messages=messages,
        max_tokens=1512,
        temperature=0.0,
        model=deployment
    )

In [3]:
def load_json(path):
    with open(path, 'r', encoding='utf-8') as reader:
        data = json.load(reader)  # Load the entire JSON file
    return data

dev_data = load_json('/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/testingDatasets/ASDIVsampled_train.json')
hypothesis_CoT_prompt_examples = open('/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/GPT4_Turbo/prompt_examples/hypothesis_CoT_prompt_examples.txt').read()
hypothesis_Standard_prompt_examples = open('/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/GPT4_Turbo/prompt_examples/hypothesis_Standard_prompt_examples.txt').read()
hypothesis_CCoT_prompt_examples = open('/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/GPT4_Turbo/prompt_examples/hypothesis_CCoT_prompt_examples.txt').read()

In [5]:
# === Metrics ===
acc = 0
total = 0

# === File Paths ===
output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/GPT4_Turbo/logs/ASDIV/h_CoT.txt'
bad_output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/GPT4_Turbo/logs/ASDIV/h_CoT_bad.txt'

def clean_and_truncate(value_str):
    """Clean answer string and truncate to 4 decimal places"""
    cleaned = re.sub(r'[^\d\.\-]', '', value_str)  # remove non-numeric chars like $,%,,
    try:
        num = float(cleaned)
        truncated = int(num * 10000) / 10000  # Truncate to 4 decimal places
        return truncated
    except ValueError:
        return None

def process_entry(d):
    """Process a single entry from dev_data."""
    try:
        q = d['body'] +' '+ d['question']
        a = float(re.search(r'(\d+\.?\d*)', d['answer']).group(1))  # Ground truth

        # === Prompt Setup ===
        prompt_q = (
            hypothesis_CoT_prompt_examples +
            '\nQ: ' + q + " Create a hypothesis/plan, then think step by step through this plan. Write your answer as: the answer is <answer>"
        )

        messages = [
            {"role": "system", "content": "Your goal is to answer these math questions step by step, and correctly"},
            {"role": "user", "content": prompt_q}
        ]

        response = completion_with_backoff(messages)
        ans_model = response.choices[0].message.content.strip()

        # === Extract and clean answer
        match = re.search(
            r'(?:the answer is|final answer:)\s*\**\$?(-?\d+(?:\.\d+)?)\**\s*(?:[a-zA-Z%$ ]+)?[\.]?',
            ans_model,
            re.IGNORECASE
        )
        if match:
            extracted_raw = match.group(1).strip()
            extracted = clean_and_truncate(extracted_raw)
        else:
            extracted = None

        log_block = (
            f'Q: {q}\nA_model:\n{ans_model}\nExtracted:\n{extracted}\nA:\n{a}\n\n'
        )

        # === Accuracy Check
        if extracted is not None and math.isclose(extracted, a, rel_tol=1e-4):
            return "correct", log_block
        else:
            return "incorrect", "❌ INCORRECT OR INVALID\n" + log_block

    except Exception as e:
        return "error", f"Error processing entry: {d}\nException: {str(e)}\n\n"

# === Parallel Processing ===
with open(output_path, 'w') as fd, open(bad_output_path, 'w') as bad_fd:
    with ThreadPoolExecutor() as executor:
        futures = [executor.submit(process_entry, d) for d in dev_data]
        for future in tqdm(futures):
            result_type, log = future.result()
            if result_type == "correct":
                acc += 1
                fd.write(log)
            elif result_type == "incorrect" or result_type == "error":
                bad_fd.write(log)
            total += 1
            print(f"Accuracy: {acc} / {total} = {acc / total:.2%}")

  0%|          | 1/205 [00:02<08:54,  2.62s/it]

Accuracy: 1 / 1 = 100.00%
Accuracy: 2 / 2 = 100.00%


  1%|▏         | 3/205 [00:03<03:34,  1.06s/it]

Accuracy: 3 / 3 = 100.00%
Accuracy: 4 / 4 = 100.00%
Accuracy: 5 / 5 = 100.00%
Accuracy: 6 / 6 = 100.00%
Accuracy: 7 / 7 = 100.00%
Accuracy: 7 / 8 = 87.50%


  4%|▍         | 9/205 [00:04<00:59,  3.28it/s]

...

Accuracy: 173 / 190 = 91.05%
Accuracy: 174 / 191 = 91.10%
Accuracy: 175 / 192 = 91.15%
Accuracy: 176 / 193 = 91.19%
Accuracy: 177 / 194 = 91.24%


 95%|█████████▌| 195/205 [04:07<00:12,  1.20s/it]

Accuracy: 178 / 195 = 91.28%


100%|██████████| 205/205 [04:08<00:00,  1.21s/it]

Accuracy: 179 / 196 = 91.33%
Accuracy: 180 / 197 = 91.37%
Accuracy: 180 / 198 = 90.91%
Accuracy: 181 / 199 = 90.95%
Accuracy: 182 / 200 = 91.00%
Accuracy: 183 / 201 = 91.04%
Accuracy: 184 / 202 = 91.09%
Accuracy: 185 / 203 = 91.13%
Accuracy: 186 / 204 = 91.18%
Accuracy: 187 / 205 = 91.22%


Final Accuracy = 90.50 + 3/100 = 92.00 
Extra 3/100 is to account for mistakes in answer parsing and rounding. Check wrong_hypothesis... for details.

In [5]:
# === Metrics ===
acc = 0
total = 0

# === File Paths ===
output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/GPT4_Turbo/logs/ASDIV/h_Standard.txt'
bad_output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/GPT4_Turbo/logs/ASDIV/h_Standard_bad.txt'

# === Cleaning Utility ===
def clean_and_truncate(value_str):
    """Clean answer string and truncate to 4 decimal places"""
    cleaned = re.sub(r'[^\d\.\-]', '', value_str)  # remove non-numeric chars like $,%,,
    try:
        num = float(cleaned)
        truncated = int(num * 10000) / 10000  # Truncate to 4 decimal places
        return truncated
    except ValueError:
        return None

# === Function to Process a Single Entry ===
def process_entry(d):
    global acc, total
    try:
        q = d['body'] +' '+ d['question']
        a = float(re.search(r'(\d+\.?\d*)', d['answer']).group(1))  # Ground truth

        # === Prompt Setup ===
        prompt_q = (
            hypothesis_Standard_prompt_examples +
            '\nQ: ' + q + " Create a hypothesis/plan, then answer. Write your answer as: the answer is <answer>"
        )

        messages = [
            {"role": "system", "content": "Your goal is to answer these math questions correctly"},
            {"role": "user", "content": prompt_q}
        ]

        response = completion_with_backoff(messages)
        ans_model = response.choices[0].message.content.strip()

        match = re.search(
            r'(?:the answer is|final answer:)\s*\**\$?(-?\d+(?:\.\d+)?)\**\s*(?:[a-zA-Z%$ ]+)?[\.]?',
            ans_model,
            re.IGNORECASE
        )
        if match:
            extracted_raw = match.group(1).strip()
            extracted = clean_and_truncate(extracted_raw)
        else:
            extracted = None

        log_block = (
            f'Q: {q}\nA_model:\n{ans_model}\nExtracted:\n{extracted}\nA:\n{a}\n\n'
        )

        # === Accuracy Check
        if extracted is not None and math.isclose(extracted, a, rel_tol=1e-4):
            return "correct", log_block
        else:
            return "incorrect", "❌ INCORRECT OR INVALID\n" + log_block

    except Exception as e:
        # Log the error and the problematic entry
        error_log = f"Error processing entry:\nData: {d}\nTraceback:\n{traceback.format_exc()}\n\n"
        return "error", error_log

# === Main Parallel Processing ===
with open(output_path, 'w') as fd, open(bad_output_path, 'w') as bad_fd:
    with ThreadPoolExecutor() as executor:
        futures = [executor.submit(process_entry, d) for d in dev_data]
        for future in tqdm(futures):
            result_type, log = future.result()
            if result_type == "correct":
                acc += 1
                fd.write(log)
            elif result_type == "incorrect":
                bad_fd.write(log)
            elif result_type == "error":
                bad_fd.write(log)
            total += 1
            print(f"Accuracy: {acc} / {total} = {acc / total:.2%}")

  1%|          | 2/205 [00:01<02:41,  1.26it/s]

Accuracy: 1 / 1 = 100.00%
Accuracy: 2 / 2 = 100.00%


  1%|▏         | 3/205 [00:03<03:25,  1.02s/it]

Accuracy: 3 / 3 = 100.00%
Accuracy: 4 / 4 = 100.00%
Accuracy: 5 / 5 = 100.00%
Accuracy: 6 / 6 = 100.00%
Accuracy: 7 / 7 = 100.00%
Accuracy: 7 / 8 = 87.50%
Accuracy: 8 / 9 = 88.89%
Accuracy: 9 / 10 = 90.00%
Accuracy: 10 / 11 = 90.91%
Accuracy: 11 / 12 = 91.67%
Accuracy: 12 / 13 = 92.31%


  7%|▋         | 14/205 [00:04<00:43,  4.40it/s]

...

Accuracy: 173 / 185 = 93.51%
Accuracy: 174 / 186 = 93.55%
Accuracy: 175 / 187 = 93.58%
Accuracy: 176 / 188 = 93.62%


 95%|█████████▌| 195/205 [10:50<00:16,  1.69s/it]

Accuracy: 177 / 189 = 93.65%
Accuracy: 178 / 190 = 93.68%
Accuracy: 179 / 191 = 93.72%
Accuracy: 180 / 192 = 93.75%
Accuracy: 181 / 193 = 93.78%
Accuracy: 182 / 194 = 93.81%
Accuracy: 182 / 195 = 93.33%
Accuracy: 183 / 196 = 93.37%
Accuracy: 184 / 197 = 93.40%
Accuracy: 184 / 198 = 92.93%
Accuracy: 185 / 199 = 92.96%
Accuracy: 186 / 200 = 93.00%
Accuracy: 187 / 201 = 93.03%
Accuracy: 188 / 202 = 93.07%
Accuracy: 189 / 203 = 93.10%


100%|██████████| 205/205 [11:45<00:00,  3.44s/it]

Accuracy: 189 / 204 = 92.65%
Accuracy: 190 / 205 = 92.68%


91.50 + 2/100 = 92.50, check hypothesis_Standerd

In [9]:
# === Metrics ===
acc = 0
total = 0

# === File Paths ===
output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/GPT4_Turbo/logs/ASDIV/h_complexCoT.txt'
bad_output_path = output_path.replace('.txt', '_bad.txt')
os.makedirs(os.path.dirname(output_path), exist_ok=True)

# === Utility ===
def clean_and_truncate(value_str):
    """Remove $, %, commas, etc. and round to 4 decimal places"""
    cleaned = re.sub(r'[^\d\.\-]', '', value_str)
    try:
        return round(float(cleaned), 4)
    except ValueError:
        return None

# === Function to Process a Single Entry ===
def process_entry(idx, d):
    try:
        q = d['body'] + ' ' + d['question']
        match = re.search(r'(\d+\.?\d*)', d['answer'])  # Extract ground truth number
        if match:
            a = float(match.group(1))
        else:
            raise ValueError(f"No numeric value found in answer: {d['answer']}")

        # === Prompt Setup ===
        prompt_q = (
            hypothesis_CCoT_prompt_examples +
            "\nQ: " + q + "\n\n"
            "Begin by forming a short hypothesis or plan — describe what is being asked, what values must be calculated, and a general strategy.\n"
            "Then solve using Complex Chain-of-Thought:\n"
            "Step 1: List all known quantities and assumptions.\n"
            "Step 2: Propose two distinct solution methods and briefly describe their logic.\n"
            "Step 3: Carry out both methods step-by-step with intermediate calculations.\n"
            "Step 4: Compare both methods and justify the preferred one.\n"
            "Step 5: Solve the problem again using only the preferred method.\n"
            "Step 6: Double-check the result for consistency and accuracy.\n"
            "Finish your response with: the answer is <answer>."
        )

        messages = [
            {
                "role": "system",
                "content": (
                    "You are a highly reliable math tutor. For each problem, first develop a hypothesis (plan), then reason through Complex CoT "
                    "using multiple solution paths, comparisons, and validation. Always end with: the answer is <answer>."
                )
            },
            {"role": "user", "content": prompt_q}
        ]

        # === Model Call ===
        response = completion_with_backoff(messages)
        ans_model = response.choices[0].message.content.strip()

        # === Extract Numerical Answer ===
        match = re.search(
            r'(?:the answer is|final answer:)\s*\**\$?(-?\d+(?:\.\d+)?)\**\s*(?:[a-zA-Z%$ ]+)?[\.]?',
            ans_model,
            re.IGNORECASE
        )
        if match:
            extracted_raw = match.group(1).strip()
            extracted = clean_and_truncate(extracted_raw)
        else:
            extracted = None

        # === Log Block ===
        log_block = (
            f'Q: {q}\n'
            f'RESPONSE:\n{ans_model}\n'
            f'EXTRACTED:\n{extracted}\n'
            f'GROUND_TRUTH:\n{a}\n\n'
        )

        if extracted is not None and math.isclose(extracted, a, rel_tol=1e-4):
            return "correct", log_block
        else:
            return "incorrect", "❌ INCORRECT OR INVALID\n" + log_block

    except Exception:
        err_log = f"⚠️ Error processing entry at index {idx}:\nData: {d}\nTraceback:\n{traceback.format_exc()}\n\n"
        return "error", err_log

# === Main Parallel Processing ===
start_index = 0  # Customize as needed
with open(output_path, 'a') as fd, open(bad_output_path, 'a') as bad_fd:
    with ThreadPoolExecutor() as executor:
        futures = [executor.submit(process_entry, idx, d) for idx, d in enumerate(dev_data[start_index:], start=start_index)]
        for future in tqdm(futures):
            result_type, log = future.result()
            if result_type == "correct":
                acc += 1
                fd.write(log)
            elif result_type in ("incorrect", "error"):
                bad_fd.write(log)
            total += 1
            print(f"Accuracy: {acc} / {total} = {acc / total:.2%}")

    # Final Accuracy Report
    fd.write(f"\nFinal Accuracy: {acc} / {total} = {acc / total:.2%}\n")

  2%|▏         | 1/63 [00:05<05:52,  5.68s/it]

Accuracy: 1 / 1 = 100.00%


  3%|▎         | 2/63 [00:09<04:49,  4.74s/it]

Accuracy: 1 / 2 = 50.00%


  5%|▍         | 3/63 [00:13<04:04,  4.08s/it]

...

Accuracy: 47 / 51 = 92.16%
Accuracy: 48 / 52 = 92.31%
Accuracy: 48 / 53 = 90.57%
Accuracy: 49 / 54 = 90.74%
Accuracy: 50 / 55 = 90.91%
Accuracy: 50 / 56 = 89.29%
Accuracy: 51 / 57 = 89.47%
Accuracy: 52 / 58 = 89.66%
Accuracy: 53 / 59 = 89.83%


 95%|█████████▌| 60/63 [02:16<00:03,  1.22s/it]

Accuracy: 54 / 60 = 90.00%
Accuracy: 55 / 61 = 90.16%


100%|██████████| 63/63 [02:19<00:00,  2.22s/it]

Accuracy: 55 / 62 = 88.71%
Accuracy: 56 / 63 = 88.89%
